In [3]:
import os
os.getcwd()


'c:\\Users\\bhara\\OneDrive\\Desktop\\mutual fund analytics'

In [4]:
os.listdir()

['.git',
 '.gitignore',
 '.ipynb_checkpoints',
 '.qodo',
 'bluestock_mf.db',
 'dashboard',
 'data',
 'dataclean+sql database.py',
 'data_dictionary.md',
 'data_dictionary_mf.db',
 'data_ingestion.py',
 'live_nav_fetch.py',
 'notebook',
 'performance_analytics.ipynb',
 'README.md',
 'recommender.py',
 'reports',
 'requirements.txt',
 'run_pipeline.py',
 'sql',
 'venv']

In [5]:
import os
os.getcwd()

'c:\\Users\\bhara\\OneDrive\\Desktop\\mutual fund analytics'

In [6]:
import os
os.getcwd()

'c:\\Users\\bhara\\OneDrive\\Desktop\\mutual fund analytics'

In [7]:
import os
os.getcwd()

'c:\\Users\\bhara\\OneDrive\\Desktop\\mutual fund analytics'

In [18]:
import pandas as pd

nav = pd.read_csv(
    r"C:\Users\bhara\OneDrive\Desktop\mutual fund analytics\data\raw\02_nav_history.csv"
)

print("NAV data loaded successfully")
print("Rows:", len(nav))
print(nav.head())

NAV data loaded successfully
Rows: 46000
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [ ]:
import os
os.listdir('data/raw')

In [ ]:
nav.head()

In [ ]:
nav = nav.sort_values(['amfi_code', 'date'])

In [ ]:
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

In [ ]:
nav.head(10)

In [ ]:
nav.head(3)

In [ ]:
(54.3474 / 54.3856) - 1

In [ ]:
nav['daily_return'].describe()

In [ ]:
nav['daily_return'].isna().sum()

In [ ]:
nav = nav.sort_values(['amfi_code', 'date'])
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
nav.head(10)

In [ ]:
nav['daily_return'].isna().sum()

In [ ]:
nav['daily_return'].describe()

In [ ]:
nav['date'] = pd.to_datetime(nav['date'])
latest_date = nav['date'].max()
latest_date

In [ ]:
three_years_ago = latest_date - pd.DateOffset(years=3)
three_years_ago

In [ ]:
latest_nav = nav.groupby('amfi_code').last().reset_index()[['amfi_code', 'nav']]
latest_nav.head()

In [ ]:
latest_nav = latest_nav.rename(columns={'nav': 'nav_end'})

In [ ]:
start_nav = (
    nav[nav['date'] <= three_years_ago]
    .groupby('amfi_code')
    .last()
    .reset_index()[['amfi_code', 'nav']]
)

start_nav = start_nav.rename(columns={'nav': 'nav_start'})
start_nav.head()

In [ ]:
cagr = latest_nav.merge(start_nav, on='amfi_code')
cagr.head()

In [ ]:
cagr['cagr_3y'] = (cagr['nav_end'] / cagr['nav_start']) ** (1/3) - 1

In [ ]:
cagr['cagr_3y_pct'] = cagr['cagr_3y'] * 100

In [ ]:
cagr[['amfi_code', 'cagr_3y_pct']].head(10)

In [ ]:
cagr = cagr.sort_values('cagr_3y_pct', ascending=False)
cagr[['amfi_code', 'cagr_3y_pct']].head(10)

In [ ]:
cagr[['amfi_code', 'cagr_3y_pct']].head(10)

In [ ]:
cagr_table = cagr_table.round(2)
cagr_table.to_csv('reports/fund_scorecard.csv', index=False)

print(cagr_table.head(10))

In [ ]:
cagr_1y = calculate_cagr(nav, 1)
cagr_1y.head()

TypeError: unsupported operand type(s) for -: 'str' and 'DateOffset'

In [12]:
def calculate_cagr(nav_df, years):
    # Latest date
    latest_date = nav_df['date'].max()

    # Date 'years' ago
    start_date = latest_date - pd.DateOffset(years=years)

    # Latest NAV for each fund
    latest_nav = (
        nav_df.groupby('amfi_code')
        .last()
        .reset_index()[['amfi_code', 'nav']]
        .rename(columns={'nav': 'nav_end'})
    )

    # NAV at or before the start date
    start_nav = (
        nav_df[nav_df['date'] <= start_date]
        .groupby('amfi_code')
        .last()
        .reset_index()[['amfi_code', 'nav']]
        .rename(columns={'nav': 'nav_start'})
    )

    # Merge start and end NAV
    cagr = latest_nav.merge(start_nav, on='amfi_code')

    # CAGR formula
    cagr[f'cagr_{years}y'] = (
        (cagr['nav_end'] / cagr['nav_start']) ** (1 / years)
    ) - 1

    # Convert to percentage
    cagr[f'cagr_{years}y_pct'] = cagr[f'cagr_{years}y'] * 100

    return cagr[['amfi_code', f'cagr_{years}y_pct']]

In [5]:
cagr_1y = calculate_cagr(nav, 1)
cagr_3y = calculate_cagr(nav, 3)
cagr_5y = calculate_cagr(nav, 5)

print(cagr_1y.head())
print(cagr_3y.head())
print(cagr_5y.head())

NameError: name 'calculate_cagr' is not defined

In [4]:
cagr_table = (
    cagr_1y
    .merge(cagr_3y, on='amfi_code')
    .merge(cagr_5y, on='amfi_code')
)

cagr_table = cagr_table.round(2)
cagr_table.head()

NameError: name 'cagr_1y' is not defined

In [ ]:
import os
os.makedirs('reports', exist_ok=True)

cagr_table.to_csv('reports/fund_scorecard.csv', index=False)

print('fund_scorecard.csv saved successfully!')

In [ ]:
# Annual risk-free rate
rf_annual = 0.065

# Convert to daily risk-free rate
rf_daily = rf_annual / 252

print(rf_daily)

In [ ]:
sharpe = (
    nav.groupby('amfi_code')['daily_return']
    .agg(['mean', 'std'])
    .reset_index()
)

sharpe['sharpe_ratio'] = (
    (sharpe['mean'] - rf_daily) / sharpe['std']
) * np.sqrt(252)

sharpe = sharpe[['amfi_code', 'sharpe_ratio']]
sharpe = sharpe.round(3)

sharpe.head()

In [ ]:
sharpe.to_csv('reports/sharpe_ratio.csv', index=False)
print('sharpe_ratio.csv saved successfully!')

In [ ]:
def calculate_sortino(group):
    returns = group['daily_return'].dropna()

    if len(returns) == 0:
        return np.nan

    downside = returns[returns < 0]

    if len(downside) == 0:
        return np.nan

    downside_std = downside.std()

    if downside_std == 0:
        return np.nan

    sortino = ((returns.mean() - rf_daily) / downside_std) * np.sqrt(252)

    return sortino

In [ ]:
sortino = (
    nav.groupby('amfi_code')
    .apply(calculate_sortino)
    .reset_index(name='sortino_ratio')
)

sortino = sortino.round(3)
sortino.head()

In [ ]:
sortino.to_csv('reports/sortino_ratio.csv', index=False)
print('sortino_ratio.csv saved successfully!')

In [ ]:
benchmark = pd.read_csv('data/raw/10_benchmark_indices.csv')
benchmark['date'] = pd.to_datetime(benchmark['date'])
benchmark.head()

In [ ]:
benchmark.columns

In [ ]:
# Keep only Nifty 100 benchmark
benchmark_100 = benchmark[benchmark['index_name'].str.contains('Nifty 100', case=False, na=False)].copy()

# Sort by date
benchmark_100 = benchmark_100.sort_values('date')

# Calculate benchmark daily returns
benchmark_100['benchmark_return'] = benchmark_100['close_value'].pct_change()

benchmark_100.head()

In [ ]:
merged = nav.merge(
    benchmark_100[['date', 'benchmark_return']],
    on='date',
    how='inner'
)

merged.head()

In [ ]:
from scipy.stats import linregress

In [ ]:
def calculate_alpha_beta(group):
    data = group[['daily_return', 'benchmark_return']].dropna()

    if len(data) < 30:
        return pd.Series({
            'alpha': np.nan,
            'beta': np.nan
        })

    result = linregress(
        data['benchmark_return'],
        data['daily_return']
    )

    beta = result.slope
    alpha = result.intercept * 252

    return pd.Series({
        'alpha': alpha,
        'beta': beta
    })

In [ ]:
alpha_beta = (
    merged.groupby('amfi_code')
    .apply(calculate_alpha_beta)
    .reset_index()
)

alpha_beta = alpha_beta.round(4)

alpha_beta.head()

In [ ]:
from scipy.stats import linregress
import pandas as pd
import numpy as np

def calculate_alpha_beta(group):
    data = group[['daily_return', 'benchmark_return']].dropna()

    if len(data) < 30:
        return pd.Series({
            'alpha': np.nan,
            'beta': np.nan
        })

    result = linregress(
        data['benchmark_return'],
        data['daily_return']
    )

    beta = result.slope
    alpha = result.intercept * 252

    return pd.Series({
        'alpha': alpha,
        'beta': beta
    })

In [ ]:
alpha_beta = merged.groupby('amfi_code').apply(calculate_alpha_beta)

alpha_beta = alpha_beta.reset_index()

alpha_beta[['alpha', 'beta']] = alpha_beta[['alpha', 'beta']].round(4)

alpha_beta.head()

In [ ]:
alpha_beta = merged.groupby('amfi_code').apply(calculate_alpha_beta)
alpha_beta = alpha_beta.reset_index()

print(alpha_beta.columns)
alpha_beta.head()

In [ ]:
from scipy.stats import linregress

results = []

for fund in merged['amfi_code'].unique():
    df = merged[merged['amfi_code'] == fund][['daily_return', 'benchmark_return']].dropna()

    if len(df) < 30:
        continue

    result = linregress(df['benchmark_return'], df['daily_return'])

    results.append({
        'amfi_code': fund,
        'alpha': result.intercept * 252,
        'beta': result.slope
    })

alpha_beta = pd.DataFrame(results)

alpha_beta = alpha_beta.round(4)

alpha_beta.head()

In [ ]:
print(merged.shape)
merged.head()

In [ ]:
benchmark['index_name'].unique()

In [ ]:
benchmark_100 = benchmark[benchmark['index_name'].str.contains('100', case=False, na=False)]

print(benchmark_100.shape)
benchmark_100['index_name'].unique()

In [ ]:
benchmark_100 = benchmark[benchmark['index_name'].str.contains('100', case=False, na=False)].copy()

benchmark_100 = benchmark_100.sort_values('date')
benchmark_100['benchmark_return'] = benchmark_100['close_value'].pct_change()

merged = nav.merge(
    benchmark_100[['date', 'benchmark_return']],
    on='date',
    how='inner'
)

print('Merged rows:', merged.shape[0])
merged.head()

In [7]:
import pandas as pd

benchmark = pd.read_csv(
    r"C:\Users\bhara\OneDrive\Desktop\mutual fund analytics\data\raw\10_benchmark_indices.csv"
)

benchmark["date"] = pd.to_datetime(benchmark["date"])

print("Benchmark loaded successfully")
print("Rows:", len(benchmark))
print(benchmark.columns.tolist())
print(benchmark.head())

Benchmark loaded successfully
Rows: 8050
['date', 'index_name', 'close_value']
        date index_name  close_value
0 2022-01-03    NIFTY50     17492.79
1 2022-01-04    NIFTY50     17689.64
2 2022-01-05    NIFTY50     17835.05
3 2022-01-06    NIFTY50     17878.51
4 2022-01-07    NIFTY50     17759.15


In [8]:
nav["date"] = pd.to_datetime(nav["date"])
print(nav[["date"]].head())

NameError: name 'nav' is not defined

In [9]:
benchmark_100 = benchmark[
    benchmark["index_name"].str.contains("100", case=False, na=False)
].copy()

benchmark_100 = benchmark_100.sort_values("date")

benchmark_100["benchmark_return"] = (
    benchmark_100["close_value"].pct_change()
)

merged = nav.merge(
    benchmark_100[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

print("Merged rows:", merged.shape[0])
print(merged.columns.tolist())

merged.head()

NameError: name 'nav' is not defined

In [10]:
nav["date"] = pd.to_datetime(nav["date"])
print(nav[["date"]].head())

NameError: name 'nav' is not defined

In [11]:
print(benchmark.columns.tolist())

['date', 'index_name', 'close_value']


In [12]:
import pandas as pd

nav = pd.read_csv(
    r"C:\Users\bhara\OneDrive\Desktop\mutual fund analytics\data\raw\02_nav_history.csv"
)

nav["date"] = pd.to_datetime(nav["date"])

print("NAV loaded successfully")
print("Rows:", len(nav))
print(nav.columns.tolist())
print(nav.head())

NAV loaded successfully
Rows: 46000
['amfi_code', 'date', 'nav']
   amfi_code       date      nav
0     119551 2022-01-03  54.3856
1     119551 2022-01-04  54.3474
2     119551 2022-01-05  54.6869
3     119551 2022-01-06  55.4550
4     119551 2022-01-07  55.3692


In [13]:
nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"].pct_change()
)

print(nav.columns.tolist())
print(nav[["amfi_code", "date", "nav", "daily_return"]].head(10))

['amfi_code', 'date', 'nav', 'daily_return']
      amfi_code       date       nav  daily_return
5750     100016 2022-01-03  520.4608           NaN
5751     100016 2022-01-04  515.0971     -0.010306
5752     100016 2022-01-05  521.7239      0.012865
5753     100016 2022-01-06  515.7880     -0.011377
5754     100016 2022-01-07  515.1639     -0.001210
5755     100016 2022-01-10  510.7136     -0.008639
5756     100016 2022-01-11  513.5542      0.005562
5757     100016 2022-01-12  512.3195     -0.002404
5758     100016 2022-01-13  510.2445     -0.004050
5759     100016 2022-01-14  514.3636      0.008073


In [ ]:
fund = merged['amfi_code'].iloc[0]

df = merged[merged['amfi_code'] == fund][['daily_return', 'benchmark_return']].dropna()

print(len(df))
df.head()

In [14]:
benchmark_100 = benchmark[
    benchmark["index_name"].str.contains("100", case=False, na=False)
].copy()

benchmark_100 = benchmark_100.sort_values("date")

benchmark_100["benchmark_return"] = (
    benchmark_100["close_value"].pct_change()
)

merged = nav.merge(
    benchmark_100[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

print("Merged rows:", len(merged))
print("Merged columns:", merged.columns.tolist())

merged.head()

Merged rows: 46000
Merged columns: ['amfi_code', 'date', 'nav', 'daily_return', 'benchmark_return']


,amfi_code,date,nav,daily_return,benchmark_return
0,100016,2022-01-03,520.4608,NaN,NaN
1,100016,2022-01-04,515.0971,-0.010306,-0.013540
2,100016,2022-01-05,521.7239,0.012865,0.004003
3,100016,2022-01-06,515.7880,-0.011377,-0.002935
4,100016,2022-01-07,515.1639,-0.001210,0.006150


In [15]:
for code in top5_codes:

    rows = merged[
        merged["amfi_code"] == code
    ][["daily_return", "benchmark_return"]].dropna()

    print(code, "->", len(rows), "matched rows")

119551 -> 1149 matched rows
120503 -> 1149 matched rows
118632 -> 1149 matched rows
119092 -> 1149 matched rows
120841 -> 1149 matched rows


In [17]:
tracking_error_list = []

for code in top5_codes:

    df = merged[
        merged["amfi_code"] == code
    ][["daily_return", "benchmark_return"]].dropna()

    if len(df) < 2:
        continue

    active_return = (
        df["daily_return"]
        - df["benchmark_return"]
    )

    te = active_return.std() * np.sqrt(252)

    tracking_error_list.append({
        "amfi_code": code,
        "tracking_error": round(te, 4)
    })

tracking_error = pd.DataFrame(tracking_error_list)

print(tracking_error)

NameError: name 'np' is not defined

In [18]:
import numpy as np

In [19]:
tracking_error_list = []

for code in top5_codes:

    df = merged[
        merged["amfi_code"] == code
    ][["daily_return", "benchmark_return"]].dropna()

    if len(df) < 2:
        continue

    active_return = (
        df["daily_return"]
        - df["benchmark_return"]
    )

    te = active_return.std() * np.sqrt(252)

    tracking_error_list.append({
        "amfi_code": code,
        "tracking_error": round(te, 4)
    })

tracking_error = pd.DataFrame(tracking_error_list)

print(tracking_error)

   amfi_code  tracking_error
0     119551          0.1912
1     120503          0.1929
2     118632          0.1921
3     119092          0.1891
4     120841          0.1831


In [20]:
tracking_error.to_csv(
    "reports/tracking_error.csv",
    index=False
)

print("tracking_error.csv saved successfully!")
print("Rows:", len(tracking_error))

tracking_error.csv saved successfully!
Rows: 5


In [21]:
std(fund_daily_return - benchmark_daily_return) * sqrt(252)

NameError: name 'std' is not defined

In [ ]:
from scipy.stats import linregress
import pandas as pd
import numpy as np

results = []

# Loop through each fund
for fund in merged['amfi_code'].unique():

    # Select one fund's returns
    df = merged.loc[
        merged['amfi_code'] == fund,
        ['daily_return', 'benchmark_return']
    ].dropna()

    # Skip if not enough data
    if len(df) < 30:
        continue

    # Regression
    reg = linregress(
        x=df['benchmark_return'],
        y=df['daily_return']
    )

    # Store alpha and beta
    results.append({
        'amfi_code': int(fund),
        'alpha': reg.intercept * 252,
        'beta': reg.slope
    })

# Create DataFrame
alpha_beta = pd.DataFrame(results)

# Round values
alpha_beta = alpha_beta.round({
    'alpha': 4,
    'beta': 4
})

print(alpha_beta.head())
print('Number of funds:', len(alpha_beta))

In [ ]:
alpha_beta.to_csv('reports/alpha_beta.csv', index=False)
print('alpha_beta.csv saved successfully!')

In [ ]:
nav = nav.sort_values(['amfi_code', 'date'])

nav['running_max'] = (
    nav.groupby('amfi_code')['nav']
    .cummax()
)

nav.head()

In [ ]:
nav['drawdown'] = (
    nav['nav'] / nav['running_max']
) - 1

nav.head()

In [ ]:
max_drawdown = (
    nav.groupby('amfi_code')['drawdown']
    .min()
    .reset_index()
)

max_drawdown = max_drawdown.rename(
    columns={'drawdown': 'max_drawdown'}
)

max_drawdown['max_drawdown_pct'] = (
    max_drawdown['max_drawdown'] * 100
).round(2)

max_drawdown.head()

In [ ]:
max_drawdown = max_drawdown.sort_values(
    'max_drawdown_pct',
    ascending=True
)

max_drawdown.head(10)

In [ ]:
max_drawdown.to_csv(
    'reports/max_drawdown.csv',
    index=False
)

print('max_drawdown.csv saved successfully!')

In [ ]:
top5 = cagr_table.sort_values(
    'cagr_3y_pct',
    ascending=False
).head(5)

top5_codes = top5['amfi_code'].tolist()

print(top5_codes)

In [ ]:
start_date = latest_date - pd.DateOffset(years=3)

fund_data = nav[
    (nav['amfi_code'].isin(top5_codes)) &
    (nav['date'] >= start_date)
].copy()

fund_data.head()

In [ ]:
fund_data['normalized_nav'] = (
    fund_data.groupby('amfi_code')['nav']
    .transform(lambda x: x / x.iloc[0] * 100)
)

fund_data.head()

In [ ]:
benchmark_3y = benchmark[
    benchmark['date'] >= start_date
].copy()

benchmark_3y['normalized_close'] = (
    benchmark_3y.groupby('index_name')['close_value']
    .transform(lambda x: x / x.iloc[0] * 100)
)

benchmark_3y.head()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot top 5 funds
for code in top5_codes:
    temp = fund_data[fund_data['amfi_code'] == code]
    plt.plot(
        temp['date'],
        temp['normalized_nav'],
        label=f'Fund {code}',
        linewidth=2
    )

# Plot benchmarks
for idx in benchmark_3y['index_name'].unique():
    temp = benchmark_3y[
        benchmark_3y['index_name'] == idx
    ]
    plt.plot(
        temp['date'],
        temp['normalized_close'],
        label=idx,
        linestyle='--',
        linewidth=2
    )

plt.title('Top 5 funds vs benchmark indices (3 years)')
plt.xlabel('Date')
plt.ylabel('Normalized value (100 = start)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(
    'reports/benchmark_comparison.png',
    dpi=300
)

plt.show()

In [3]:
print(top5_codes)

NameError: name 'top5_codes' is not defined

In [1]:
print("Top 5 codes:")
print(top5_codes)

print("\nMerged shape:")
print(merged.shape)

print("\nMerged columns:")
print(merged.columns.tolist())

Top 5 codes:


NameError: name 'top5_codes' is not defined

In [2]:
for code in top5_codes:
    df_test = merged[merged["amfi_code"] == code][
        ["daily_return", "benchmark_return"]
    ].dropna()

    print(code, "->", len(df_test), "matched rows")

NameError: name 'top5_codes' is not defined

In [4]:
top5_codes = [
    119551,  # SBI Bluechip
    120503,  # ICICI Bluechip
    118632,  # Nippon Large Cap
    119092,  # Axis Bluechip
    120841   # Kotak Bluechip
]

print("Top 5 codes:", top5_codes)

Top 5 codes: [119551, 120503, 118632, 119092, 120841]


In [5]:
print(merged.shape)

NameError: name 'merged' is not defined

In [6]:
top5_codes = [119551, 120503, 118632, 119092, 120841]
print(top5_codes)

[119551, 120503, 118632, 119092, 120841]


In [ ]:
tracking_error = []

for code in top5_codes:

    df = merged[merged['amfi_code'] == code][
        ['daily_return', 'benchmark_return']
    ].dropna()

    te = (
        (df['daily_return'] - df['benchmark_return']).std()
        * np.sqrt(252)
    )

    tracking_error.append({
        'amfi_code': code,
        'tracking_error': round(te, 4)
    })

tracking_error = pd.DataFrame(tracking_error)

tracking_error.to_csv(
    'reports/tracking_error.csv',
    index=False
)

tracking_error

In [ ]:
print('Results length:', len(results))
print(alpha_beta.shape)
alpha_beta.head()

In [ ]:
print('Merged shape:', merged.shape)
print('Unique funds:', merged['amfi_code'].nunique())

fund = merged['amfi_code'].iloc[0]
df = merged[merged['amfi_code'] == fund][['daily_return', 'benchmark_return']].dropna()

print('Rows for first fund:', len(df))
print(df.head())

In [ ]:
from scipy.stats import linregress

alpha_list = []

for fund in merged['amfi_code'].unique():
    df = merged[merged['amfi_code'] == fund][['daily_return', 'benchmark_return']].dropna()

    reg = linregress(df['benchmark_return'], df['daily_return'])

    alpha_list.append({
        'amfi_code': fund,
        'alpha': round(reg.intercept * 252, 4),
        'beta': round(reg.slope, 4)
    })

alpha_beta = pd.DataFrame(alpha_list)

print(alpha_beta.shape)
alpha_beta.head()

In [ ]:
alpha_beta.head(10)

In [ ]:
alpha_beta.to_csv('reports/alpha_beta.csv', index=False)
print('alpha_beta.csv saved successfully!')

In [1]:
final_scorecard = (
    cagr_table
    .merge(sharpe, on='amfi_code')
    .merge(sortino, on='amfi_code')
    .merge(alpha_beta, on='amfi_code')
    .merge(max_drawdown[['amfi_code', 'max_drawdown_pct']], on='amfi_code')
)

final_scorecard.head()

NameError: name 'cagr_table' is not defined

In [ ]:
final_scorecard = (
    cagr_table
    .merge(sharpe, on='amfi_code')
    .merge(sortino, on='amfi_code')
    .merge(alpha_beta, on='amfi_code')
    .merge(max_drawdown[['amfi_code', 'max_drawdown_pct']], on='amfi_code')
)

final_scorecard.head()

In [ ]:
# Higher CAGR is better
final_scorecard['rank_cagr'] = final_scorecard['cagr_3y_pct'].rank(ascending=False)

# Higher Sharpe is better
final_scorecard['rank_sharpe'] = final_scorecard['sharpe_ratio'].rank(ascending=False)

# Higher Alpha is better
final_scorecard['rank_alpha'] = final_scorecard['alpha'].rank(ascending=False)

# Less negative drawdown is better
final_scorecard['rank_dd'] = final_scorecard['max_drawdown_pct'].rank(ascending=False)

final_scorecard.head()

In [ ]:
n = len(final_scorecard)

# Convert ranks into 0-100 scores
for col in ['rank_cagr', 'rank_sharpe', 'rank_alpha', 'rank_dd']:
    score_col = col.replace('rank_', 'score_')
    final_scorecard[score_col] = (
        (n - final_scorecard[col]) / (n - 1)
    ) * 100

# Weighted composite score
final_scorecard['fund_score'] = (
    0.30 * final_scorecard['score_cagr'] +
    0.25 * final_scorecard['score_sharpe'] +
    0.20 * final_scorecard['score_alpha'] +
    0.10 * final_scorecard['score_dd']
)

final_scorecard = final_scorecard.round(2)

final_scorecard.head()

In [ ]:
final_scorecard = final_scorecard.sort_values(
    'fund_score',
    ascending=False
)

final_scorecard[['amfi_code',
                 'cagr_3y_pct',
                 'sharpe_ratio',
                 'alpha',
                 'max_drawdown_pct',
                 'fund_score']].head(10)

In [ ]:
final_scorecard.to_csv(
    'reports/fund_scorecard.csv',
    index=False
)

print('Final fund_scorecard.csv saved successfully!')

In [ ]:
top5_codes = final_scorecard.head(5)['amfi_code'].tolist()

start_date = latest_date - pd.DateOffset(years=3)

fund_data = nav[
    (nav['amfi_code'].isin(top5_codes)) &
    (nav['date'] >= start_date)
].copy()

fund_data['normalized_nav'] = (
    fund_data.groupby('amfi_code')['nav']
    .transform(lambda x: x / x.iloc[0] * 100)
)

benchmark_3y = benchmark[
    benchmark['date'] >= start_date
].copy()

benchmark_3y['normalized_close'] = (
    benchmark_3y.groupby('index_name')['close_value']
    .transform(lambda x: x / x.iloc[0] * 100)
)

plt.figure(figsize=(12,6))

for code in top5_codes:
    temp = fund_data[fund_data['amfi_code'] == code]
    plt.plot(temp['date'],
             temp['normalized_nav'],
             label=f'Fund {code}')

for idx in benchmark_3y['index_name'].unique():
    temp = benchmark_3y[benchmark_3y['index_name'] == idx]
    plt.plot(temp['date'],
             temp['normalized_close'],
             label=idx,
             linestyle='--')

plt.title('Top 5 funds vs benchmark indices (3 years)')
plt.xlabel('Date')
plt.ylabel('Normalized value (100 = start)')
plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    'reports/benchmark_comparison.png',
    dpi=300
)

plt.show()

print('benchmark_comparison.png saved successfully!')

In [ ]:
# Save final fund scorecard
final_scorecard.to_csv(
    "reports/fund_scorecard.csv",
    index=False
)

print("fund_scorecard.csv saved successfully!")
print("Rows:", len(final_scorecard))

In [17]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")

print("NAV data loaded successfully")
print("Rows:", len(nav))
print(nav.head())

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/02_nav_history.csv'

In [20]:
print(calculate_cagr)

<function calculate_cagr at 0x000002082C6C5A60>


In [21]:
import pandas as pd
import numpy as np

def calculate_cagr(nav_data, years):
    df = nav_data.copy()

    df["date"] = pd.to_datetime(df["date"])

    results = []

    for amfi_code, group in df.groupby("amfi_code"):

        group = group.sort_values("date")

        start_date = group["date"].min()
        end_date = group["date"].max()

        target_start = end_date - pd.DateOffset(years=years)

        period_data = group[group["date"] >= target_start]

        if len(period_data) < 2:
            continue

        start_nav = period_data.iloc[0]["nav"]
        end_nav = period_data.iloc[-1]["nav"]

        actual_days = (
            period_data.iloc[-1]["date"] -
            period_data.iloc[0]["date"]
        ).days

        if start_nav > 0 and actual_days > 0:
            cagr = (end_nav / start_nav) ** (365 / actual_days) - 1
        else:
            cagr = np.nan

        results.append({
            "amfi_code": amfi_code,
            f"cagr_{years}y_pct": cagr * 100
        })

    return pd.DataFrame(results)

In [22]:
cagr_1y = calculate_cagr(nav, 1)

print("1-Year CAGR calculated successfully")
print("Rows:", len(cagr_1y))

cagr_1y.head()

1-Year CAGR calculated successfully
Rows: 40


,amfi_code,cagr_1y_pct
0,100016,-2.224271
1,100025,3.704969
2,100033,53.232396
3,101206,47.924120
4,101207,-23.986032


In [23]:
cagr_3y = calculate_cagr(nav, 3)
cagr_5y = calculate_cagr(nav, 5)

print("3-Year CAGR calculated successfully")
print("3-Year rows:", len(cagr_3y))

print("\n5-Year CAGR calculated successfully")
print("5-Year rows:", len(cagr_5y))

print("\n3-Year CAGR:")
display(cagr_3y.head())

print("\n5-Year CAGR:")
display(cagr_5y.head())

3-Year CAGR calculated successfully
3-Year rows: 40

5-Year CAGR calculated successfully
5-Year rows: 40

3-Year CAGR:


,amfi_code,cagr_3y_pct
0,100016,1.291462
1,100025,3.912748
2,100033,32.408510
3,101206,28.937763
4,101207,-4.148672



5-Year CAGR:


,amfi_code,cagr_5y_pct
0,100016,2.635246
1,100025,4.455091
2,100033,30.099704
3,101206,23.520489
4,101207,7.933121


In [24]:
cagr_table = (
    cagr_1y
    .merge(cagr_3y, on="amfi_code", how="outer")
    .merge(cagr_5y, on="amfi_code", how="outer")
)

print("CAGR table created successfully")
print("Rows:", len(cagr_table))

display(cagr_table.head())

CAGR table created successfully
Rows: 40


,amfi_code,cagr_1y_pct,cagr_3y_pct,cagr_5y_pct
0,100016,-2.224271,1.291462,2.635246
1,100025,3.704969,3.912748,4.455091
2,100033,53.232396,32.408510,30.099704
3,101206,47.924120,28.937763,23.520489
4,101207,-23.986032,-4.148672,7.933121


In [25]:
print(cagr_table.columns.tolist())
print(cagr_table.shape)
print(cagr_table.isna().sum())

['amfi_code', 'cagr_1y_pct', 'cagr_3y_pct', 'cagr_5y_pct']
(40, 4)
amfi_code      0
cagr_1y_pct    0
cagr_3y_pct    0
cagr_5y_pct    0
dtype: int64


In [26]:
import pandas as pd

sharpe = pd.read_csv("reports/sharpe_ratio.csv")
sortino = pd.read_csv("reports/sortino_ratio.csv")
alpha_beta = pd.read_csv("reports/alpha_beta.csv")
max_drawdown = pd.read_csv("reports/max_drawdown.csv")

print("Sharpe:", sharpe.shape)
print("Sortino:", sortino.shape)
print("Alpha/Beta:", alpha_beta.shape)
print("Max Drawdown:", max_drawdown.shape)

Sharpe: (40, 2)
Sortino: (40, 2)
Alpha/Beta: (40, 3)
Max Drawdown: (40, 3)


In [27]:
final_scorecard = (
    cagr_table
    .merge(sharpe, on="amfi_code", how="left")
    .merge(sortino, on="amfi_code", how="left")
    .merge(alpha_beta, on="amfi_code", how="left")
    .merge(
        max_drawdown[["amfi_code", "max_drawdown_pct"]],
        on="amfi_code",
        how="left"
    )
)

print("Final scorecard created")
print("Rows:", len(final_scorecard))
print("Columns:", final_scorecard.columns.tolist())

display(final_scorecard.head())

Final scorecard created
Rows: 40
Columns: ['amfi_code', 'cagr_1y_pct', 'cagr_3y_pct', 'cagr_5y_pct', 'sharpe_ratio', 'sortino_ratio', 'alpha', 'beta', 'max_drawdown_pct']


,amfi_code,cagr_1y_pct,cagr_3y_pct,cagr_5y_pct,sharpe_ratio,sortino_ratio,alpha,beta,max_drawdown_pct
0,100016,-2.224271,1.291462,2.635246,-0.202,-0.351,0.0375,-0.0583,-24.73
1,100025,3.704969,3.912748,4.455091,-0.567,-0.942,0.0428,0.0012,-4.31
2,100033,53.232396,32.408510,30.099704,1.094,1.829,0.2720,0.0051,-16.22
3,101206,47.924120,28.937763,23.520489,1.027,1.800,0.2140,0.0211,-11.29
4,101207,-23.986032,-4.148672,7.933121,0.163,0.277,0.1090,-0.0653,-35.45


In [28]:
print(final_scorecard.isna().sum())

amfi_code           0
cagr_1y_pct         0
cagr_3y_pct         0
cagr_5y_pct         0
sharpe_ratio        0
sortino_ratio       0
alpha               0
beta                0
max_drawdown_pct    0
dtype: int64


In [29]:
final_scorecard.to_csv(
    "reports/fund_scorecard.csv",
    index=False
)

print("fund_scorecard.csv saved successfully!")
print("Rows:", len(final_scorecard))

fund_scorecard.csv saved successfully!
Rows: 40


In [30]:
print(final_scorecard.shape)
print(final_scorecard.columns.tolist())

(40, 9)
['amfi_code', 'cagr_1y_pct', 'cagr_3y_pct', 'cagr_5y_pct', 'sharpe_ratio', 'sortino_ratio', 'alpha', 'beta', 'max_drawdown_pct']


In [31]:
print(final_scorecard.isna().sum())

amfi_code           0
cagr_1y_pct         0
cagr_3y_pct         0
cagr_5y_pct         0
sharpe_ratio        0
sortino_ratio       0
alpha               0
beta                0
max_drawdown_pct    0
dtype: int64


In [32]:
final_scorecard.sort_values(
    "sharpe_ratio",
    ascending=False
).head(10)

,amfi_code,cagr_1y_pct,cagr_3y_pct,cagr_5y_pct,sharpe_ratio,sortino_ratio,alpha,beta,max_drawdown_pct
34,148567,20.360678,33.965137,30.949920,1.448,2.386,0.2698,0.0237,-11.27
30,120843,26.657082,29.552134,30.883326,1.307,2.364,0.2733,-0.0228,-12.97
36,148569,39.751761,29.148729,31.924486,1.235,2.147,0.2827,0.0181,-16.40
19,119551,60.437341,30.424882,25.784921,1.208,2.140,0.2320,-0.0318,-15.01
25,120505,29.604659,31.744363,32.801599,1.180,2.029,0.2926,0.0005,-18.19
38,149323,21.481222,26.842444,29.558105,1.132,1.875,0.2660,-0.0025,-17.25
2,100033,53.232396,32.408510,30.099704,1.094,1.829,0.2720,0.0051,-16.22
9,118632,33.981048,22.629512,24.031196,1.082,1.850,0.2183,-0.0084,-17.41
3,101206,47.924120,28.937763,23.520489,1.027,1.800,0.2140,0.0211,-11.29
24,120504,13.064279,32.453426,23.277448,1.027,1.805,0.2119,0.0162,-12.59


In [33]:
final_scorecard.sort_values(
    "cagr_5y_pct",
    ascending=False
).head(10)

,amfi_code,cagr_1y_pct,cagr_3y_pct,cagr_5y_pct,sharpe_ratio,sortino_ratio,alpha,beta,max_drawdown_pct
25,120505,29.604659,31.744363,32.801599,1.180,2.029,0.2926,0.0005,-18.19
21,119598,82.776059,26.642601,32.398084,0.945,1.675,0.3034,-0.0232,-28.71
39,149324,65.138719,26.972733,32.262108,0.950,1.620,0.3006,0.0115,-31.17
36,148569,39.751761,29.148729,31.924486,1.235,2.147,0.2827,0.0181,-16.40
34,148567,20.360678,33.965137,30.949920,1.448,2.386,0.2698,0.0237,-11.27
30,120843,26.657082,29.552134,30.883326,1.307,2.364,0.2733,-0.0228,-12.97
2,100033,53.232396,32.408510,30.099704,1.094,1.829,0.2720,0.0051,-16.22
38,149323,21.481222,26.842444,29.558105,1.132,1.875,0.2660,-0.0025,-17.25
16,119094,22.261065,35.074709,28.192608,0.998,1.704,0.2608,-0.0663,-20.96
19,119551,60.437341,30.424882,25.784921,1.208,2.140,0.2320,-0.0318,-15.01


In [34]:
final_scorecard.sort_values(
    "max_drawdown_pct",
    ascending=False
).head(10)

,amfi_code,cagr_1y_pct,cagr_3y_pct,cagr_5y_pct,sharpe_ratio,sortino_ratio,alpha,beta,max_drawdown_pct
27,120507,7.425051,7.387296,7.232411,0.496,1.052,0.0675,-0.0004,-0.10
31,120844,7.120323,6.689662,6.910380,-0.089,-0.188,0.0646,-0.0004,-0.12
5,101208,7.236645,6.309844,6.504430,-0.816,-1.681,0.0609,0.0003,-0.16
1,100025,3.704969,3.912748,4.455091,-0.567,-0.942,0.0428,0.0012,-4.31
18,119120,5.521996,5.834877,5.885271,-0.227,-0.377,0.0562,-0.0064,-4.33
13,118636,10.454940,4.058424,5.313155,-0.357,-0.614,0.0507,0.0013,-8.32
6,102885,20.207704,19.647660,18.218826,0.817,1.436,0.1705,-0.0195,-10.86
34,148567,20.360678,33.965137,30.949920,1.448,2.386,0.2698,0.0237,-11.27
3,101206,47.924120,28.937763,23.520489,1.027,1.800,0.2140,0.0211,-11.29
12,118635,22.493225,19.988239,15.945481,0.665,1.138,0.1514,-0.0014,-11.65
